## Energy Data Cleaning & Analysis Notebook 

This notebook processes and validates hourly electricity load data for various regions, focusing on the PJM interconnection. The workflow includes:

- **Data Loading and Cleaning:**  
    The code reads raw hourly load data from `energyData/pjm_hourly_est.csv`, identifies the datetime column, and processes each region's data. For each region:
    - It extracts the relevant columns, converts the datetime, sorts, and resamples the data to daily averages.
    - The cleaned daily data is saved as a CSV file in the `energyData_clean` directory, with filenames indicating the region and aggregation level.

- **Validation of Cleaned Data:**  
    The notebook then iterates through all cleaned daily CSV files in `energyData_clean`, loading each into a DataFrame. For each file:
    - It checks that the index is a datetime type and that there are no missing values.
    - It verifies that the load values fall within a reasonable range (between 5,000 and 100,000 MW).
    - It prints summary statistics, including the number of rows, date range, and MW range for each region.

This process ensures that the cleaned datasets are consistent, complete, and ready for further analysis or visualization.

In [16]:
import pandas as pd
import os

# Folder setup
data_dir = "energyData"
cleaned_dir = "energyData_clean"
os.makedirs(cleaned_dir, exist_ok=True)

# Only process pjm_hourly_est.csv
filepath = os.path.join(data_dir, "pjm_hourly_est.csv")
df = pd.read_csv(filepath)
print(f"Processing pjm_hourly_est.csv...")
print("Columns:", df.columns.tolist())

datetime_col = 'Datetime' if 'Datetime' in df.columns else df.columns[0]

# Process each region column
for region_col in df.columns:
    if region_col == datetime_col:
        continue

    df_region = df[[datetime_col, region_col]].dropna()
    df_region[datetime_col] = pd.to_datetime(df_region[datetime_col], errors='coerce')
    df_region.set_index(datetime_col, inplace=True)
    df_region = df_region.sort_index()

    df_daily = df_region.resample('D').mean()
    df_daily.dropna(inplace=True)

    # Normalize PJM_Load → PJM
    region_name = "PJM" if region_col == "PJM_Load" else region_col
    cleaned_path = os.path.join(cleaned_dir, f"{region_name}_MW_hourly_daily.csv")
    df_daily.to_csv(cleaned_path)
    print(f"Saved cleaned file to: {cleaned_path}")


Processing pjm_hourly_est.csv...
Columns: ['Datetime', 'AEP', 'COMED', 'DAYTON', 'DEOK', 'DOM', 'DUQ', 'EKPC', 'FE', 'NI', 'PJME', 'PJMW', 'PJM_Load']
Saved cleaned file to: energyData_clean\AEP_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\COMED_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\DAYTON_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\DEOK_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\DOM_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\DUQ_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\EKPC_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\FE_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\NI_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\PJME_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\PJMW_MW_hourly_daily.csv
Saved cleaned file to: energyData_clean\PJM_MW_hourly_daily.csv


In [17]:
file_list = [f for f in os.listdir(cleaned_dir) if f.endswith("_daily.csv")]

for filename in file_list:
    filepath = os.path.join(cleaned_dir, filename)
    print(f"\nValidating {filename}...")

    try:
        df = pd.read_csv(filepath, index_col=0, parse_dates=True)

        # Basic checks
        assert isinstance(df.index, pd.DatetimeIndex), "Index is not datetime"
        assert df.isnull().sum().sum() == 0, "Contains missing values"

        # Range checks
        col = df.columns[0]
        if not (5000 < df[col].min() and df[col].max() < 100000):
            print(f" {col} values out of expected MW range: min={df[col].min()}, max={df[col].max()}")

        # Print info
        print(f"{filename}:")
        print(f"   Rows: {len(df)}")
        print(f"   Date Range: {df.index.min().date()} → {df.index.max().date()}")
        print(f"   MW Range: {df[col].min():.2f} → {df[col].max():.2f}")

    except Exception as e:
        print(f"Error in {filename}: {e}")


Validating AEP_MW_hourly_daily.csv...
AEP_MW_hourly_daily.csv:
   Rows: 5055
   Date Range: 2004-10-01 → 2018-08-03
   MW Range: 11078.04 → 22847.88

Validating COMED_MW_hourly_daily.csv...
COMED_MW_hourly_daily.csv:
   Rows: 2772
   Date Range: 2011-01-01 → 2018-08-03
   MW Range: 8148.75 → 19920.29

Validating DAYTON_MW_hourly_daily.csv...
 DAYTON values out of expected MW range: min=1366.3333333333333, max=3136.625
DAYTON_MW_hourly_daily.csv:
   Rows: 5055
   Date Range: 2004-10-01 → 2018-08-03
   MW Range: 1366.33 → 3136.62

Validating DEOK_MW_hourly_daily.csv...
 DEOK values out of expected MW range: min=1219.0, max=4503.458333333333
DEOK_MW_hourly_daily.csv:
   Rows: 2407
   Date Range: 2012-01-01 → 2018-08-03
   MW Range: 1219.00 → 4503.46

Validating DOM_MW_hourly_daily.csv...
DOM_MW_hourly_daily.csv:
   Rows: 4843
   Date Range: 2005-05-01 → 2018-08-03
   MW Range: 7772.00 → 18976.62

Validating DUQ_MW_hourly_daily.csv...
 DUQ values out of expected MW range: min=1188.1666666

In [18]:
summary = []
for filename in file_list:
    region = filename.split('_')[0]
    df = pd.read_csv(os.path.join(cleaned_dir, filename), index_col=0, parse_dates=True)
    summary.append({'Region': region, 'Entries': len(df)})

for entry in summary:
    print(f"Region: {entry['Region']}, Rows: {entry['Entries']}")

# Final rows count 
total_rows = sum(entry['Entries'] for entry in summary)
print(f"\nTotal rows across all regions: {total_rows}")


Region: AEP, Rows: 5055
Region: COMED, Rows: 2772
Region: DAYTON, Rows: 5055
Region: DEOK, Rows: 2407
Region: DOM, Rows: 4843
Region: DUQ, Rows: 4963
Region: EKPC, Rows: 1890
Region: FE, Rows: 2621
Region: NI, Rows: 2437
Region: PJME, Rows: 6059
Region: PJMW, Rows: 5969
Region: PJM, Rows: 1372

Total rows across all regions: 45443
